# Agentic RAG


# 1. Setup

In [20]:
from uretriever.configs.api_config import get_env
from uretriever.configs.path_config import DATA_DIR, MODEL_DIR, PROMPT_DIR


MODEL_CONFIG = {
    "provider": "llama_cpp",
    "model_path": str(MODEL_DIR / "mistral-7b-instruct-v0.2.Q4_K_M.gguf"),
    "max_tokens": 512,
    "n_ctx": 4096,
    "temperature": 0,
    "n_threads": 4,
    "n_gpu_layers": -1,
    'verbose': False

    # "provider": "openai",
    # "model": "gpt-4.1",
    # "base_url": get_env("GITHUB_MODEL_API"),
    # "api_key": get_env("GITHUB_TOKEN"),
    # "max_tokens": 512,
    # "temperature": 0,
    # 'verbose': False
}

RAW_DATA_DIR = DATA_DIR / "raw"
INDEX_DATA_DIR = DATA_DIR / "processed"
INDEX_DATA_DIR.mkdir(parents=True, exist_ok=True)

# CSV corpus files for index building
LAWS_CSV_PATH = RAW_DATA_DIR / "laws_de.csv"
COURTS_CSV_PATH = RAW_DATA_DIR / "court_considerations.csv"
# Index cache file
LAWS_INDEX_PATH = INDEX_DATA_DIR / "laws_bm25_index.pkl"
COURTS_INDEX_PATH = INDEX_DATA_DIR / "courts_bm25_index.pkl"
FORCE_REBUILD = False

# Retrieval settings
TOP_K_LAWS = 40
TOP_K_COURTS = 40

print(f"RAW_DATA_DIR: {RAW_DATA_DIR}")
print(f"INDEX_DATA_DIR: {INDEX_DATA_DIR}")
print(f"PROMPT_DIR: {PROMPT_DIR}")

print(f"LAWS_CSV_PATH: {LAWS_CSV_PATH}")
print(f"COURTS_CSV_PATH: {COURTS_CSV_PATH}")
print(f"LAWS_INDEX_PATH: {LAWS_INDEX_PATH}")
print(f"COURTS_INDEX_PATH: {COURTS_INDEX_PATH}")

RAW_DATA_DIR: D:\URetriever\data\raw
INDEX_DATA_DIR: D:\URetriever\data\processed
PROMPT_DIR: D:\URetriever\prompts
LAWS_CSV_PATH: D:\URetriever\data\raw\laws_de.csv
COURTS_CSV_PATH: D:\URetriever\data\raw\court_considerations.csv
LAWS_INDEX_PATH: D:\URetriever\data\processed\laws_bm25_index.pkl
COURTS_INDEX_PATH: D:\URetriever\data\processed\courts_bm25_index.pkl


# 2. Build or Load BM25 Indices

In [21]:
# Load or build BM25 index
from uretriever.BM25Index import get_or_build_index


laws_index = get_or_build_index(
    name="laws",
    csv_path=LAWS_CSV_PATH,
    index_path=LAWS_INDEX_PATH,
    force_rebuild=FORCE_REBUILD,
)
print(f"\nLaws index: {len(laws_index.documents):,} documents")

Loading cached laws index from D:\URetriever\data\processed\laws_bm25_index.pkl
  Loaded 175,933 documents

Laws index: 175,933 documents


In [22]:
# Test search
test_results = laws_index.search("Vertrag", top_k=3)
print(f"\nTest search 'Vertrag': {len(test_results)} results")
if test_results:
    print(f"  Top result: {test_results[0].get('citation', 'N/A')}")


Test search 'Vertrag': 3 results
  Top result: Art. 17 Abs. 6 VID


In [23]:
# Load or build courts index

courts_index = get_or_build_index(
    name="courts",
    csv_path=COURTS_CSV_PATH,
    index_path=COURTS_INDEX_PATH,
    force_rebuild=FORCE_REBUILD,
    max_rows=100000  # Change to use bigger corpus
)
print(f"\nCourts index: {len(courts_index.documents):,} documents")

Loading cached courts index from D:\URetriever\data\processed\courts_bm25_index.pkl
  Loaded 100,000 documents

Courts index: 100,000 documents


In [24]:
# Test search
test_results = courts_index.search("Meinungsfreiheit", top_k=3)
print(f"\nTest search 'Meinungsfreiheit': {len(test_results)} results")
if test_results:
    print(f"  Top result: {test_results[0].get('citation', 'N/A')}")


Test search 'Meinungsfreiheit': 3 results
  Top result: BGE 148 I 33 E. 6.1


# 3. Define Search Tools

In [25]:
from langchain_core.tools import tool


def make_law_search_tool(
    index,
    top_k: int = 5,
    max_excerpt_length: int = 300,
):
    """Factory that returns a @tool-decorated law search function."""

    @tool
    def search_laws(query: str) -> str:
        """Search Swiss federal laws (SR/Systematische Rechtssammlung) by keywords.

        Input: Search query string (can be in German, French, Italian, or English)
        Output: List of relevant law citations with text excerpts
        Use this tool to find relevant federal law provisions for a legal question.
        Example queries: 'contract formation requirements', 'Vertragsabschluss', 'divorce grounds'
        """
        if not query or not query.strip():
            return "Error: Empty query. Please provide search terms."

        results = index.search(query, top_k=top_k)
        if not results:
            return f"No relevant federal laws found for: '{query}'"

        formatted = []
        for doc in results:
            citation = doc.get("citation", "Unknown")
            text = doc.get("text", "")
            if len(text) > max_excerpt_length:
                text = text[:max_excerpt_length] + "..."
            formatted.append(f"- {citation}: {text}")

        return "\n".join(formatted)

    return search_laws


def make_court_search_tool(
    index,
    top_k: int = 5,
    max_excerpt_length: int = 300,
):
    """Factory that returns a @tool-decorated court search function."""

    @tool
    def search_courts(query: str) -> str:
        """Search Swiss Federal Court decisions by keywords.

        Input: Search query string (German, French, Italian, or English)
        Output: List of relevant court decision citations with excerpts
        Use this tool to find relevant case law and judicial interpretations.
        Example queries: 'negligence standard of care', 'Sorgfaltspflicht', 'contract interpretation'
        """
        if not query or not query.strip():
            return "Error: Empty query. Please provide search terms."

        results = index.search(query, top_k=top_k)
        if not results:
            return f"No relevant court decisions found for: '{query}'"

        formatted = []
        for doc in results:
            citation = doc.get("citation", "Unknown")
            text = doc.get("text", "")
            if len(text) > max_excerpt_length:
                text = text[:max_excerpt_length] + "..."
            formatted.append(f"- {citation}: {text}")

        return "\n".join(formatted)

    return search_courts


law_tool = make_law_search_tool(
    index=laws_index,
    top_k=TOP_K_LAWS,
    max_excerpt_length=300,
)

court_tool = make_court_search_tool(
    index=courts_index,
    top_k=TOP_K_COURTS,
    max_excerpt_length=300,
)

TOOLS = [law_tool, court_tool]

print("Tools:")
for t in TOOLS:
    print(f"  - {t.name}: {t.description.splitlines()[0]}")

Tools:
  - search_laws: Search Swiss federal laws (SR/Systematische Rechtssammlung) by keywords.
  - search_courts: Search Swiss Federal Court decisions by keywords.


In [26]:
# Test tools
print("Testing law search:")
print(law_tool.invoke("Vertrag Abschluss"))

print("\nTesting court search:")
print(court_tool.invoke("Meinungsfreiheit"))

Testing law search:
- Art. 22 Abs. 1 OR: 1 Durch Vertrag kann die Verpflichtung zum Abschluss eines künftigen Vertrages begründet werden.
- Art. 23 OR: Der Vertrag ist für denjenigen unverbindlich, der sich beim Abschluss in einem wesentlichen Irrtum befunden hat.
- Art. 20 Abs. 2 VEAGOG: 2 Der Leistungsauftrag wird mittels Vertrag erteilt. Es besteht kein Rechtsanspruch auf den Abschluss eines Leistungsauftrags.47
- Art. 22 Abs. 3 VEAGOG: 3 Der Leistungsauftrag wird mittels Vertrag erteilt. Es besteht kein Rechtsanspruch auf den Abschluss eines Leistungsauftrags.52
- Art. 166 Abs. 2 BV: 2 Sie genehmigt die völkerrechtlichen Verträge; ausgenommen sind die Verträge, für deren Abschluss auf Grund von Gesetz oder völkerrechtlichem Vertrag der Bundesrat zuständig ist.
- Art. 152 Abs. 3bis ParlG: 3bis Der Bundesrat konsultiert die zuständigen Kommissionen, bevor er:a. einen völkerrechtlichen Vertrag vorläufig anwendet, dessen Abschluss oder Änderung durch die Bundesversammlung genehmigt wer

# 4. Load LLM

In [27]:
# Load chat model
from uretriever.chat_model_loader import load_chat_model


provider = MODEL_CONFIG["provider"]
model_config = {k: v for k, v in MODEL_CONFIG.items() if k != "provider"}
chat_model = load_chat_model(provider, **model_config)

llama_new_context_with_model: n_batch is less than GGML_KQ_MASK_PAD - increasing to 32
llama_new_context_with_model: n_ctx_per_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


## 5. Define ReAct Agent

In [28]:
# Load prompt
from uretriever.utils import load_text


system_prompt = load_text(PROMPT_DIR / "system_agent.txt")
example_prompt = load_text(PROMPT_DIR / "examples_agent.txt")

print("\n-- system_prompt --\n")
print(system_prompt)

print("\n-- example_prompt --\n")
print(example_prompt)


-- system_prompt --

You are a Swiss legal research assistant with access to two search tools:

1. search_laws(query): Search Swiss federal legislation (SR/Systematic Collection of Federal Law)
   - Returns relevant statutory provisions with citations and text excerpts
   - Use for statutory law: codes, acts, ordinances

2. search_courts(query): Search Swiss Federal Supreme Court decisions (BGE)
   - Returns relevant case law with citations and excerpts
   - Use for court decisions and precedents

Your task is to find the most relevant Swiss legal citations for the user's question.

Language rules:
- Users usually ask questions in English.
- The legal database is in German.
- You must reformulate the user's question into proper German legal search terminology.
- Always use German in tool calls.

Research rules:
- Search BOTH: legislation AND court decisions
- Use multiple search queries with German legal terms
- Keep calling the tools until all relevant sources have been found

Output

In [29]:
from langchain.agents import create_agent

agent = create_agent(
    model=chat_model,
    tools=TOOLS,
    system_prompt=system_prompt + example_prompt,
)

In [ ]:
from langchain_core.messages import HumanMessage

# Test agent with a sample query
test_query = "What are the requirements for a valid contract under Swiss law?"

response = agent.invoke(
    {
        "messages": [HumanMessage(content=test_query)],
    }
)
last_message = response["messages"][-1]
last_message.content